# Tiny decision classifier (EmbeddingBag, no torchtext)

A fastText-style baseline for the decision detection task. It uses a hashed bag of words with `nn.EmbeddingBag` pooling and a single linear head to stay extremely small. No `torchtext` dependencies.


In [ ]:
%load_ext autoreload
%autoreload 2


## Imports and config

- Uses deterministic hashing for tokens to keep a fixed vocab size without building a vocab file.
- Model size is controlled by `vocab_size` and `embed_dim` (for example, 20k vocab and 64-dim embeddings is roughly 1.3M parameters).


In [ ]:
import hashlib
import os
import random
import re
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from torch import nn
from torch.cuda.amp import autocast, GradScaler
from torch.utils.data import DataLoader, Dataset
from unidecode import unidecode

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

device = "cuda" if torch.cuda.is_available() else "cpu"


@dataclass
class Config:
    data_path: str = "sentences-decision-manual.csv"
    vocab_size: int = 20000
    embed_dim: int = 64
    max_tokens: int = 128
    batch_size: int = 512
    lr: float = 5e-3
    weight_decay: float = 1e-3
    epochs: int = 50
    dropout: float = 0.1
    num_workers: int = 2
    checkpoint_path: str = "checkpoints/tiny-embeddingbag.pt"


cfg = Config()

print(f"device: {device}")


## Load data

Input columns: `path`, `nro_registro`, `tomo`, `sentence`, `decision`, `hace_lugar`.


In [ ]:
data = pd.read_csv(
    cfg.data_path,
    usecols=["path", "nro_registro", "tomo", "sentence", "decision", "hace_lugar"],
)
print(data.head())
print(f"Loaded {len(data)} rows")


## Preprocess labels

- Drop null rows and duplicate sentences.
- Collapse labels to a binary target: `0 = not decision`, `1 = decision` (regardless of `hace_lugar`).


In [ ]:
data.dropna(inplace=True)


def force_bool(value):
    return True if value in ["True", True, 1, "1"] else False


def get_category(row):
    decision, hace_lugar = row
    if not decision:
        return 0
    if decision and not hace_lugar:
        return 1
    if decision and hace_lugar:
        return 1
    raise ValueError("unexpected label combo")


data["decision"] = data["decision"].apply(force_bool).astype(bool)
data["hace_lugar"] = data["hace_lugar"].apply(force_bool).astype(bool)
data["category"] = data[["decision", "hace_lugar"]].apply(get_category, axis=1)

data.dropna(subset=["category"], inplace=True)
data.drop_duplicates(subset="sentence", inplace=True)

print(f"After cleaning: {len(data)} rows")
print(data[["category"]].value_counts(normalize=True) * 100)


## Train/val/test split

80/10/10 with stratification on the binary label.


In [ ]:
train_df, test_df = train_test_split(
    data, test_size=0.2, random_state=SEED, stratify=data["category"]
)
test_df, val_df = train_test_split(
    test_df, test_size=0.5, random_state=SEED, stratify=test_df["category"]
)

for name, df_ in {"train": train_df, "val": val_df, "test": test_df}.items():
    print(
        f"{name}: {len(df_)} rows | class balance: {df_['category'].value_counts(normalize=True).to_dict()}"
    )


## Tokenization and dataset

- Simple normalize → whitespace split.
- Deterministic 32-bit Blake2 hash per token to map into `[0, vocab_size)`.
- Truncate to `max_tokens` to bound memory.


In [ ]:
def normalize_text(text: str) -> str:
    text = unidecode(str(text)).lower()
    text = re.sub(r"\s+", " ", text).strip()
    return text


def tokenize(text: str) -> list[str]:
    text = normalize_text(text)
    return [tok for tok in text.split(" ") if tok]


def hash_token(token: str, vocab_size: int) -> int:
    digest = hashlib.blake2b(token.encode("utf-8"), digest_size=4).digest()
    return int.from_bytes(digest, "little") % vocab_size


def encode_text(text: str, cfg: Config) -> torch.Tensor:
    tokens = tokenize(text)
    token_ids = [hash_token(tok, cfg.vocab_size) for tok in tokens[: cfg.max_tokens]]
    if not token_ids:
        token_ids = [0]
    return torch.tensor(token_ids, dtype=torch.long)


class HashedTextDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, cfg: Config):
        self.texts = frame["sentence"].tolist()
        self.labels = frame["category"].astype(int).tolist()
        self.cfg = cfg

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx: int):
        token_ids = encode_text(self.texts[idx], self.cfg)
        label = self.labels[idx]
        return token_ids, label


def collate_batch(batch):
    token_seqs, labels = zip(*batch)
    offsets = torch.zeros(len(batch), dtype=torch.long)
    total = 0
    flat_tokens = []
    for i, tokens in enumerate(token_seqs):
        offsets[i] = total
        flat_tokens.append(tokens)
        total += len(tokens)
    flat_tokens = torch.cat(flat_tokens)
    labels = torch.tensor(labels, dtype=torch.long)
    return flat_tokens, offsets, labels


train_ds = HashedTextDataset(train_df, cfg)
val_ds = HashedTextDataset(val_df, cfg)
test_ds = HashedTextDataset(test_df, cfg)

train_loader = DataLoader(
    train_ds,
    batch_size=cfg.batch_size,
    shuffle=True,
    collate_fn=collate_batch,
    num_workers=cfg.num_workers,
)
val_loader = DataLoader(
    val_ds,
    batch_size=cfg.batch_size,
    shuffle=False,
    collate_fn=collate_batch,
    num_workers=cfg.num_workers,
)
test_loader = DataLoader(
    test_ds,
    batch_size=cfg.batch_size,
    shuffle=False,
    collate_fn=collate_batch,
    num_workers=cfg.num_workers,
)


## Model: EmbeddingBag + Linear

Tiny architecture with mean pooling. Dropout is optional for slight regularization.


In [ ]:
class TinyEmbeddingBagClassifier(nn.Module):
    def __init__(self, cfg: Config, num_classes: int = 2):
        super().__init__()
        self.embedding = nn.EmbeddingBag(cfg.vocab_size, cfg.embed_dim, mode="mean")
        self.dropout = nn.Dropout(cfg.dropout)
        self.head = nn.Linear(cfg.embed_dim, num_classes)

    def forward(self, tokens, offsets):
        x = self.embedding(tokens, offsets)
        x = self.dropout(x)
        return self.head(x)


model = TinyEmbeddingBagClassifier(cfg).to(device)
param_count = sum(p.numel() for p in model.parameters())
print(model)
print(f"Parameter count: {param_count:,}")


## Training utilities

- Class-weighted cross entropy to compensate imbalance.
- Simple train/val loop with best-model tracking.


In [ ]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_df["category"]),
    y=train_df["category"],
)
class_weights = torch.tensor(class_weights, dtype=torch.float, device=device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(
    model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=1,
    threshold=1e-3,
    min_lr=1e-5,
)
scaler = GradScaler(enabled=(device == "cuda"))


def run_epoch(model, loader, optimizer=None, scaler=None):
    train_mode = optimizer is not None
    model.train() if train_mode else model.eval()

    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    for tokens, offsets, labels in loader:
        tokens = tokens.to(device)
        offsets = offsets.to(device)
        labels = labels.to(device)

        if train_mode:
            optimizer.zero_grad(set_to_none=True)
            with autocast(enabled=device == "cuda"):
                logits = model(tokens, offsets)
                loss = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            with torch.no_grad(), autocast(enabled=device == "cuda"):
                logits = model(tokens, offsets)
                loss = criterion(logits, labels)

        preds = logits.argmax(dim=1)
        total_correct += (preds == labels).sum().item()
        total_loss += loss.item() * labels.size(0)
        total_examples += labels.size(0)

    avg_loss = total_loss / total_examples
    acc = total_correct / total_examples
    return avg_loss, acc


best_state = None
best_val_loss = float("inf")

for epoch in range(cfg.epochs):
    train_loss, train_acc = run_epoch(model, train_loader, optimizer, scaler=scaler)
    val_loss, val_acc = run_epoch(model, val_loader, optimizer=None, scaler=None)

    lr_before = optimizer.param_groups[0]["lr"]
    scheduler.step(val_loss)
    lr_after = optimizer.param_groups[0]["lr"]

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {k: v.cpu() for k, v in model.state_dict().items()}

    if lr_after < lr_before and best_state is not None:
        # resume from best snapshot whenever LR is reduced
        model.load_state_dict(best_state)

    cur_lr = optimizer.param_groups[0]["lr"]
    print(
        f"Epoch {epoch + 1:02d} | lr {cur_lr:.2e} | train loss {train_loss:.4f} acc {train_acc:.3f} | "
        f"val loss {val_loss:.4f} acc {val_acc:.3f}"
    )

if best_state is not None:
    model.load_state_dict(best_state)
    print(f"Loaded best state with val loss={best_val_loss:.4f}")


## Evaluation

Compute accuracy and a quick classification report on the held-out splits.


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix


def collect_predictions(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for tokens, offsets, labels in loader:
            tokens = tokens.to(device)
            offsets = offsets.to(device)
            logits = model(tokens, offsets)
            preds = logits.argmax(dim=1).cpu().tolist()
            all_preds.extend(preds)
            all_labels.extend(labels.tolist())
    return all_labels, all_preds


for split_name, loader in [("val", val_loader), ("test", test_loader)]:
    y_true, y_pred = collect_predictions(model, loader)
    print(f"=== {split_name.upper()} ===")
    print(classification_report(y_true, y_pred, digits=3))
    print("Confusion matrix:")
    print(confusion_matrix(y_true, y_pred))


## Save checkpoint

Saves a lightweight `state_dict` for later reuse.


In [ ]:
from pathlib import Path


ckpt_path = Path(cfg.checkpoint_path)
ckpt_path.parent.mkdir(parents=True, exist_ok=True)
torch.save({"config": cfg.__dict__, "state_dict": model.state_dict()}, ckpt_path)
print(f"Saved to {ckpt_path}")


## Save as safetensors


In [ ]:
from safetensors.torch import save_file
import json

# Save model weights as safetensors
safetensor_path = ckpt_path.with_suffix(".safetensors")
save_file(model.state_dict(), safetensor_path)

# Save config separately as JSON
config_path = ckpt_path.with_suffix(".json")
with open(config_path, "w") as f:
    json.dump(cfg.__dict__, f, indent=2)

print(f"Saved model to {safetensor_path}")
print(f"Saved config to {config_path}")